# Wykrywanie botów

### Pełen zbiór pr_comments

In [ ]:
import pandas as pd

# Wczytanie pr_comments.parquet
pr_comments = pd.read_parquet("Data/pr_comments.parquet")

# Filtr user_type=="User"
user_filter = pr_comments['user_type'] == "User"

# Filtr potencjalnych botow po nazwie uzytkownika
bot_like_filter = pr_comments['user'].str.contains(r'bot|ci|copilot|ai|automation', case=False, na=False)

# Laczymy filtry
suspected_bots = pr_comments[user_filter & bot_like_filter]

# Liczymy liczbe komentarzy dla kazdego uzytkownika
bot_counts = suspected_bots['user'].value_counts()

# Filtrujemy tych, ktorzy maja minimum 10 komentarzy
bot_counts_10plus = bot_counts[bot_counts >= 10]

# Sortujemy rekordy po ilosci komentarzy
bot_counts_10plus_sorted = bot_counts_10plus.sort_values(ascending=False)

# Wyswietlamy 50 pierwszych podejrzanych
display(bot_counts_10plus.head(50))

# Znajdz Boty i przestaw na BotOther
bots_users = ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot', 'facebook-github-bot','pulumi-bot','raycastbot']
pr_comments.loc[pr_comments['user'].isin(bots_users), 'user_type'] = 'BotOther'

user
captainsafia       52
jakebailey         48
ian-at-airbyte     34
NoritakaIkeda      32
k8s-ci-robot       28
MayaRainer         27
wolfSSL-Bot        25
vdaas-ci           25
jairmyree          15
hairyhum           13
raineorshine       13
takenagain         12
pytorchmergebot    11
aishwaryabh        11
owidbot            10
Name: count, dtype: int64

### Pełen zbiór pr_comments - słabszy filtr bots_filter

In [2]:
filter = pr_comments['user_type']=='User'
bots_filter = pr_comments['user'].str.contains("bot")

# TODO używając np tych kolumn
display(pr_comments[filter& bots_filter]['user'].value_counts())
display(pr_comments[filter]['user'].value_counts()[:50])

# TODO: Znajdź Boty i przestaw na BotOther
bots_users =  ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot','facebook-github-bot','pulumi-bot','raycastbot']

pr_comments[pr_comments['user'].isin(bots_users)]

pr_comments.loc[pr_comments['user'].isin(bots_users), 'user_type'] = 'BotOther'

user
metamaskbot                   9
k8s-infra-cherrypick-robot    4
zwave-js-bot                  4
Issues-translate-bot          2
slangbot                      2
gravity-ui-bot                2
botberry                      2
gradio-pr-bot                 2
nf-core-bot                   2
mm-cloud-bot                  1
sre-ci-robot                  1
pyvista-bot                   1
throbotaith                   1
wheelsbot7                    1
istio-policy-bot              1
bits-bot                      1
Name: count, dtype: int64

user
joaomdmoura                            264
davidfowl                              161
anupriya13                             153
PureWeen                               144
slavingia                              132
coveralls                              129
benbalter                              108
vs-mobiletools-engineering-service2     86
mitchdenny                              84
marrobi                                 79
harupy                                  70
tomhrr                                  70
pierrevalade                            69
parseplatformorg                        67
jcstein                                 64
transphorm                              58
shige                                   56
pelikhan                                53
T-Gro                                   52
captainsafia                            52
brianrob                                52
JoshLove-msft                           51
meganrogge                              50
aarons

# Ciche PR

In [ ]:
# Wczytanie wszystkich PRów z pełnego zbioru
all_pr_df = pd.read_parquet("Data/pull_request.parquet")

# Lista wszystkich
all_pr_ids = all_pr_df["id"].unique()

# PRy które mają komentarze
prs_with_comments = pr_comments["pr_id"].unique()

# PRy bez komentarzy
prs_no_comments = set(all_pr_ids) - set(prs_with_comments)

len(prs_no_comments)

20621

# Podział PR reviews ze względu na to kto robi recenzję

In [4]:
types_per_pr = (
    pr_comments.groupby("pr_id")["user_type"]
      .apply(lambda s: tuple(sorted(set(s))))
      .reset_index(name="type_set")
)

# 2) zlicz PR-y dla każdego zestawu typów
distribution = types_per_pr["type_set"].value_counts()

print(distribution)

print("\n\n# Recenzje bez ludzkiego czynnika")
distribution[distribution.index.map(lambda t: 'User' in t)].sum()/distribution.sum()

type_set
(Bot,)                   7625
(Bot, User)              3749
(User,)                  1066
(BotOther,)               191
(Bot, BotOther, User)     143
(Bot, BotOther)           142
(BotOther, User)           59
Name: count, dtype: int64


# Recenzje bez ludzkiego czynnika


np.float64(0.38666666666666666)

# Code review zrobiony przez ludzi dla PR przygotowanych przez AI


In [5]:
#pr_comments

import re

AGENT_PATTERN = r"(copilot|coderabbit|coderabbitai|bugbot|gemini|claude|devin|" \
                r"sourcery-ai|greptile|giselles-ai|giselle|pytorchbot|pytorchmergebot|sentry|zwave-js-bot|create-docs-giselles-ai|dotnet-policy-service)"

COMMAND_PATTERN = (
    r"(run|retry|recheck|rebase|review|summari[sz]e|start|continue|fix|resolve|"\
    r"merge|test|format|lint|sign|add|remove|update|rerun|check|verify|resume|"\
    r"generate|create|solve|clean|commit|pack|try|thanks|conflicts)"
)

AGENT_CMD_RE = re.compile(
    rf"""(?:^|\s)
    ([@\\\/]?{AGENT_PATTERN})
    [^.\n]*?
    \b{COMMAND_PATTERN}\b
    |
    \b{COMMAND_PATTERN}\b
    [^.\n]*?
    ([@\/\\]?{AGENT_PATTERN})
    """,
    re.IGNORECASE | re.VERBOSE,
)

STARTS_WITH_AGENT = re.compile(
    rf"""
    ^[@/ ]?{AGENT_PATTERN}
    """,
    re.IGNORECASE | re.VERBOSE,
)

OTHER_COMMANDS = re.compile(
    r"^[/#!@\[\\](azp|check-enforcer|easycla|run|rebase|build-deploy-preview|backport|autoformat|test|bump-version|ci|ba-|format-|poe|"\
    r"cherrypick|huppy|poetry-|ok-|queue-|ci-|fix-|engineer|deploypr|update-|format|checkenforcer|approve-|review|gogoeditdiff"\
    r"|microsoft-github-policy-service|intelci|gpu|spinwick)",
    re.IGNORECASE | re.VERBOSE,
)
OTHER_COMMANDS_EXPL = re.compile(
    r"^(<\!-- buildkite-pr-bot-trigger-comment -->|\*\*\[chatops:help\]\*\*|\n\[\!\[coverage status\]|pre-commit\.ci autofix)", re.IGNORECASE)


WORKFLOW = re.compile(r"^\s*("\
    "resolve merge conflicts"\
    "|can one of the admins verify this patch?"\
    "|lgtm!?"\
    "|thank you!"\
    "|try again"\
    "|recheck"\
    "|i have read the cla document and i hereby sign the cla"\
    "|duplicate"\
    "|\!sync"\
    "|thanks"\
    "|fix lint issue"\
    "|ok"\
    "|closing as stale"\
    "|done"\
    "|thx"\
    ")\s*", 
    re.IGNORECASE)

AUTO_BOT = re.compile(
    rf"""
    ^\[approvalnotifier\]
    |^\s*hi\s+[@/]?{AGENT_PATTERN}\.\s+thanks\s+for\s+your\s+pr\..*
    |^\s*\*\*?\[WARNING:CONFLICT\]\*\*?.*
    |^\s*\*\*?\[FORMAT\]\*\*?.*
    |^\s*\*\*?API change check\*\*?.*
    |\:?tada\:
    """,
    re.IGNORECASE | re.VERBOSE,
)

def _matches(pattern, text: str) -> bool:
    return isinstance(text, str) and bool(pattern.search(text))


def is_agentic_ai_command(text: str) -> bool:
    return _matches(AGENT_CMD_RE, text) or _matches(STARTS_WITH_AGENT, text)

def is_other_command(text: str) -> bool:
    return _matches(OTHER_COMMANDS, text)

def is_auto_bot(text: str) -> bool:
    return _matches(AUTO_BOT, text) or _matches(OTHER_COMMANDS_EXPL, text)

def is_workflow(text: str) -> bool:
    return _matches(WORKFLOW, text)


def label_rev_type(text: str) -> str:
    if is_agentic_ai_command(text):
        return "agentic_cmd"
    if is_other_command(text) or is_auto_bot(text):
        return "bot"
    if is_workflow(text):
        return "workflow"
    return "review"

In [6]:
tests = [
    "/gemini review",
    "/azp",
    "/easycla",
    "ala ma kota",
    "@copilot try again",
    "try again @copilot",
    "[approvalnotifier]",
    "hi @copilot. thanks for your pr.",
    "#gogoeditdiff",
    "@microsoft-github-policy-service",
    "<!-- buildkite-pr-bot-trigger-comment -->",
    "**[chatops:help]**",
    "\n[![coverage status]",
    "lgtm",
    "pre-commit.ci autofix",
    "!sync",
    "ok    ",
    "can one of the admins verify this patch?    ",
    "lgtm!",
    "/intelci: run",
    "@pytorchmergebot merge",
    "!gpu",
    "**[WARNING:CONFLICT]**",
    "\\azp",
    "comment :tada:",
    
]

for t in tests:
    print(t,"=>", label_rev_type(t))


/gemini review => agentic_cmd
/azp => bot
/easycla => bot
ala ma kota => review
@copilot try again => agentic_cmd
try again @copilot => agentic_cmd
[approvalnotifier] => bot
hi @copilot. thanks for your pr. => bot
#gogoeditdiff => bot
@microsoft-github-policy-service => bot
<!-- buildkite-pr-bot-trigger-comment --> => bot
**[chatops:help]** => bot

[![coverage status] => bot
lgtm => workflow
pre-commit.ci autofix => bot
!sync => workflow
ok     => workflow
can one of the admins verify this patch?     => workflow
lgtm! => workflow
/intelci: run => bot
@pytorchmergebot merge => agentic_cmd
!gpu => bot
**[WARNING:CONFLICT]** => bot
\azp => bot
comment :tada: => bot


In [7]:
m_comments = pr_comments["user_type"] == "User"

pr_comments.loc[m_comments,"rev_type"] = pr_comments.loc[m_comments, "body"].apply(label_rev_type)

In [8]:
sum(pr_comments["user_type"] == "Bot")

27416

In [9]:
pr_comments['rev_type'].value_counts() / sum(m_comments)

rev_type
review         0.616568
agentic_cmd    0.283684
bot            0.070966
workflow       0.028782
Name: count, dtype: float64

In [10]:
pr_comments['rev_type'].value_counts()

rev_type
review         6855
agentic_cmd    3154
bot             789
workflow        320
Name: count, dtype: int64

In [11]:
sum(m_comments)

11118

In [12]:
display(pr_comments['rev_type'].value_counts())
print(sum(m_comments))
len(pr_comments) - sum(m_comments)

rev_type
review         6855
agentic_cmd    3154
bot             789
workflow        320
Name: count, dtype: int64

11118


28004

In [13]:
pr_comments[pr_comments['rev_type'] == 'bot']

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
339,3126788393,3269032263,buger,14009,User,2025-07-28T11:28:19Z,/review how do you feel about this changes? Ar...,bot
424,2875765090,3057825906,lindy65,13020833,User,2025-05-13T09:38:34Z,/spinwick create --env MM_FEATUREFLAGS_Experim...,bot
747,3121064306,3264933329,sre-ci-robot,56469371,User,2025-07-26T02:59:06Z,[APPROVALNOTIFIER] This PR is **NOT APPROVED**...,bot
752,3048720914,3212218608,coveralls,2354108,User,2025-07-08T12:23:06Z,\n[![Coverage Status](https://coveralls.io/bui...,bot
755,3076621858,3212218608,nilo19,36728755,User,2025-07-16T03:41:50Z,/cherrypick release-1.30,bot
...,...,...,...,...,...,...,...,...
37353,2692276496,2888672043,natikgadzhi,43633,User,2025-03-01T15:12:26Z,/format-fix\r\n\n> Format-fix job started... [...,bot
38345,2658231239,2852623041,natikgadzhi,43633,User,2025-02-14T04:21:51Z,"/bump-version type=""patch"" changelog=""Enable t...",bot
38618,2692333438,2888982802,natikgadzhi,43633,User,2025-03-01T17:38:57Z,"/bump-version type=""minor"" changelog=""Update t...",bot
38687,2601756081,2798574828,Nikhil-Nandagopal,3897254,User,2025-01-20T08:38:04Z,/build-deploy-preview skip-tests=true\r\n\r\n,bot


In [14]:
display(pr_comments['pr_id'].unique())

array([3107321792, 3234660269, 3037457814, ..., 2761443934, 3047296744,
       2823956710], shape=(12975,))

In [15]:
pr_comments[(pr_comments['rev_type'] == 'review') & (pr_comments['user_type'] == 'User')]

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
2,2848667986,3037457814,wilsonccccc,6146503,User,2025-05-03T15:12:37Z,"Hi, thanks for sharing and contributing! Pleas...",review
4,3013998830,3132410695,razimantv,3823215,User,2025-06-27T18:16:05Z,Can you rebase it to the current master? Pleas...,review
7,2720368428,2915784381,nickskalkin,3934579,User,2025-03-13T08:30:47Z,Co-authored-by: Claude \r\n😁,review
9,2721609147,2915784381,damassi,236943,User,2025-03-13T15:09:45Z,> Actually rewrite that commit and remove me a...,review
15,3111630135,3256172444,nilo19,36728755,User,2025-07-24T01:13:29Z,If it's hard to review I can split it into mul...,review
...,...,...,...,...,...,...,...,...
39107,2564708592,2762228850,shreysingla11,62649026,User,2024-12-29T12:25:49Z,"### Code Review Summary\n\nOverall, this PR ma...",review
39110,2693663048,2890563538,joaomdmoura,667063,User,2025-03-03T08:52:22Z,Disclaimer: This review was made by a crew of ...,review
39113,2564173971,2761443934,joaomdmoura,667063,User,2024-12-28T04:37:43Z,Disclaimer: This review was made by a crew of ...,review
39118,2874995243,3047296744,retrogtx,76801862,User,2025-05-13T04:07:09Z,failing tests + does not solve the original is...,review


In [16]:
def pr_summary(labels):
    unique = set(labels)
    return str(unique)

pr_labels_comment = (
    pr_comments[(pr_comments['user_type'] == 'User')].groupby("pr_id")["rev_type"]
      .apply(pr_summary)
      .reset_index(name="pr_type")
)

In [17]:
counts_comment = pr_labels_comment["pr_type"].value_counts()
print(counts_comment)

pr_type
{'review'}                               2887
{'agentic_cmd'}                           900
{'review', 'agentic_cmd'}                 507
{'bot'}                                   198
{'bot', 'review'}                         130
{'workflow'}                              126
{'review', 'workflow'}                     97
{'bot', 'agentic_cmd'}                     60
{'bot', 'review', 'agentic_cmd'}           59
{'review', 'workflow', 'agentic_cmd'}      27
{'workflow', 'agentic_cmd'}                14
{'bot', 'review', 'workflow'}              10
{'bot', 'workflow'}                         2
Name: count, dtype: int64


In [18]:
# No classical human part
print(counts_comment[~counts_comment.index.str.contains("review")].sum()/counts_comment.sum())

# Only clasical
clasical_filter = counts_comment.index.str.contains("{'review'}")
print(counts_comment[clasical_filter].sum()/counts_comment.sum())

# Classical + other
print(counts_comment[counts_comment.index.str.contains("review") & ~clasical_filter].sum()/counts_comment.sum())

0.25911899541558703
0.575443492126769
0.16543751245764401


In [19]:
filtered_comments = pr_comments[
    (pr_comments["user_type"] == "User") &
    (pr_comments["rev_type"] == "review") 
]

top_bodies_comments = (
    filtered_comments["body"].str.lower()
    .value_counts()
    .head(50)     
)

print(top_bodies_comments)

body
looks like your session timed out. try again.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [20]:
# narzedzie do wyszukiwania podejrzanych danych
filtered_comments[filtered_comments['body'].str.lower().str.contains("/spinwick create")]

,id,pr_id,user,user_id,user_type,created_at,body,rev_type


In [21]:
slash_counts_comments = (
    pr_comments[pr_comments["body"].str.startswith("/", na=False) & (pr_comments["rev_type"] == 'review')]["body"].value_counts()
)
print(slash_counts_comments)

body
/hold                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [22]:
mask_comments = pr_comments["body"].str.startswith("/", na=False) & (pr_comments["rev_type"] == "review")

# extract first token after '/'
first_word_comments = pr_comments.loc[mask_comments, "body"].str.extract(r"^/(\S+)", expand=False)

slash_counts_comments = first_word_comments.value_counts()

print(slash_counts_comments)

body
hold                                                                                                   1
lgtm                                                                                                   1
close                                                                                                  1
cvt                                                                                                    1
mnt/vss/_work/1/s/src/Areas/Storage/Services/StorageService.cs(436,1):                                 1
Users/runner/work/1/s/tests/Areas/Storage/UnitTests/DataLake/File/FileUploadCommandTests.cs(94,66):    1
build                                                                                                  1
Users/builder/azdo/_work/3/s/src/Controls/tests/Core.UnitTests/ContentViewUnitTest.cs(477,26):         1
azurepipelines                                                                                         1
version-bump                                      

In [23]:
pr_comments[pr_comments['rev_type'] == 'agentic_cmd']

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
1,3090154270,3234660269,Fank,1900106,User,2025-07-18T17:22:34Z,"claude budget reached, development is on hold ...",agentic_cmd
23,3051558554,3214876564,wtfsayo,82053242,User,2025-07-09T07:50:45Z,bugbot run,agentic_cmd
24,3051563644,3214876564,wtfsayo,82053242,User,2025-07-09T07:52:15Z,@claude @coderabbitai full review,agentic_cmd
27,3051613653,3214876564,wtfsayo,82053242,User,2025-07-09T08:09:07Z,@claude @coderabbitai full review,agentic_cmd
32,3051663813,3214876564,wtfsayo,82053242,User,2025-07-09T08:26:10Z,@claude @coderabbitai full review\r\n\r\nbugbo...,agentic_cmd
...,...,...,...,...,...,...,...,...
38608,2914500275,3094261129,yamadashy,5019072,User,2025-05-28T00:21:09Z,@claude review,agentic_cmd
38610,2914540681,3094261129,yamadashy,5019072,User,2025-05-28T00:50:51Z,@coderabbitai review,agentic_cmd
38611,2914540916,3094261129,yamadashy,5019072,User,2025-05-28T00:51:01Z,/gemini review,agentic_cmd
38839,2646962123,2841314055,MH4GF,31152321,User,2025-02-10T05:34:39Z,"Devin, can you add some more here?\r\nhttps://...",agentic_cmd
